# Peer-Graded Assignment: Course Project
## IBM: Statistics for Data Science with Python

### Task 1: Import Libraries and Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

# Cài đặt hiển thị đồ thị đẹp hơn
sns.set(style="whitegrid")

In [ ]:
# Đọc dữ liệu từ URL bài tập
url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ST0151EN-SkillsNetwork/labs/boston_housing.csv'
boston_df = pd.read_csv(url)

# Kiểm tra 5 dòng đầu tiên
boston_df.head()

### Task 2: Generate Descriptive Statistics and Visualizations

#### 1. Boxplot cho "Median value of owner-occupied homes" (MEDV)

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(y='MEDV', data=boston_df, color='skyblue')
plt.title('Boxplot of Median Value of Owner-Occupied Homes (MEDV)')
plt.ylabel('Median Value in $1000s')
plt.show()

#### 2. Bar plot cho biến Charles river (CHAS)

In [ ]:
plt.figure(figsize=(7, 5))
sns.countplot(x='CHAS', data=boston_df, palette='Set2')
plt.title('Bar Plot of Charles River Variable (CHAS)')
plt.xlabel('Tract Bounds Charles River (0 = No, 1 = Yes)')
plt.ylabel('Count')
plt.show()

#### 3. Boxplot so sánh MEDV theo các nhóm tuổi (AGE)

In [ ]:
# Phân chia 3 nhóm tuổi
boston_df.loc[(boston_df['AGE'] <= 35), 'age_group'] = '35 years and younger'
boston_df.loc[(boston_df['AGE'] > 35) & (boston_df['AGE'] < 70), 'age_group'] = 'between 35 and 70 years'
boston_df.loc[(boston_df['AGE'] >= 70), 'age_group'] = '70 years and older'

plt.figure(figsize=(10, 6))
order_groups = ['35 years and younger', 'between 35 and 70 years', '70 years and older']
sns.boxplot(x='age_group', y='MEDV', data=boston_df, order=order_groups, palette='Pastel1')
plt.title('Median Value of Homes (MEDV) across Age Groups')
plt.xlabel('Age Group')
plt.ylabel('Median Value in $1000s')
plt.show()

#### 4. Scatter plot giữa Nitric oxide (NOX) và Non-retail business (INDUS)

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(x='NOX', y='INDUS', data=boston_df, alpha=0.7, color='teal')
plt.title('Relationship between Nitric Oxide Concentrations and Non-Retail Business Acres')
plt.xlabel('Nitric Oxides Concentration (parts per 10 million)')
plt.ylabel('Proportion of Non-Retail Business Acres per Town')
plt.show()

#### 5. Histogram cho tỷ lệ học sinh trên giáo viên (PTRATIO)

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(boston_df['PTRATIO'], bins=15, kde=False, color='coral')
plt.title('Histogram of Pupil-Teacher Ratio by Town (PTRATIO)')
plt.xlabel('Pupil-Teacher Ratio')
plt.ylabel('Frequency')
plt.show()

### Task 3: Statistical Tests
Mức ý nghĩa thống kê áp dụng: $\alpha = 0.05$

#### Test 1: T-test cho hai mẫu độc lập (Independent Samples T-Test)
**Câu hỏi:** Có sự khác biệt đáng kể về giá trị nhà (MEDV) giữa khu vực giáp sông Charles (CHAS=1) và không giáp sông (CHAS=0) hay không?
- **H0:** $\mu_1 = \mu_2$ (Không có sự khác biệt về trung bình giá nhà giữa hai khu vực).
- **H1:** $\mu_1 \neq \mu_2$ (Có sự khác biệt đáng kể về trung bình giá nhà giữa hai khu vực).

In [ ]:
# Kiểm tra tính đồng nhất của phương sai bằng Levene's test
chas_0 = boston_df[boston_df['CHAS'] == 0]['MEDV']
chas_1 = boston_df[boston_df['CHAS'] == 1]['MEDV']

levene_stat, levene_p = scipy.stats.levene(chas_0, chas_1, center='mean')
print(f"Levene's test p-value: {levene_p:.5f}")

# T-test (nếu Levene's test p < 0.05 thì equal_var=False)
t_stat, t_pval = scipy.stats.ttest_ind(chas_0, chas_1, equal_var=(levene_p > 0.05))
print(f"T-statistic: {t_stat:.4f}, p-value: {t_pval:.5e}")

if t_pval < 0.05:
    print("Kết luận: Bác bỏ H0. Có sự khác biệt có ý nghĩa thống kê về giá nhà giữa khu vực giáp sông và không giáp sông.")
else:
    print("Kết luận: Chưa đủ bằng chứng bác bỏ H0.")

#### Test 2: ANOVA (Analysis of Variance)
**Câu hỏi:** Có sự khác biệt về giá nhà (MEDV) giữa các nhóm tuổi công trình (AGE) hay không?
- **H0:** $\mu_1 = \mu_2 = \mu_3$ (Trung bình MEDV như nhau giữa 3 nhóm tuổi).
- **H1:** Ít nhất một nhóm có trung bình MEDV khác biệt.

In [ ]:
g1 = boston_df[boston_df['age_group'] == '35 years and younger']['MEDV']
g2 = boston_df[boston_df['age_group'] == 'between 35 and 70 years']['MEDV']
g3 = boston_df[boston_df['age_group'] == '70 years and older']['MEDV']

f_stat, anova_p = scipy.stats.f_oneway(g1, g2, g3)
print(f"ANOVA F-statistic: {f_stat:.4f}, p-value: {anova_p:.5e}")

if anova_p < 0.05:
    print("Kết luận: Bác bỏ H0. Có sự khác biệt có ý nghĩa thống kê về giá trị nhà ở giữa các nhóm tuổi.")
else:
    print("Kết luận: Chưa đủ bằng chứng bác bỏ H0.")

#### Test 3: Pearson Correlation
**Câu hỏi:** Có mối tương quan giữa nồng độ Nitric oxide (NOX) và tỷ lệ diện tích đất công nghiệp không bán lẻ (INDUS) không?
- **H0:** Hệ số tương quan $r = 0$ (Không có tương quan tuyến tính).
- **H1:** Hệ số tương quan $r \neq 0$ (Có tương quan tuyến tính).

In [ ]:
corr, corr_p = scipy.stats.pearsonr(boston_df['NOX'], boston_df['INDUS'])
print(f"Pearson correlation coefficient: {corr:.4f}, p-value: {corr_p:.5e}")

if corr_p < 0.05:
    print("Kết luận: Bác bỏ H0. Có tương quan tuyến tính thuận chiều, rất mạnh giữa NOX và INDUS.")
else:
    print("Kết luận: Chưa đủ bằng chứng bác bỏ H0.")

#### Test 4: Phân tích Hồi quy Tuyến tính (Regression Analysis)
**Câu hỏi:** Khoảng cách đến 5 trung tâm việc làm tại Boston (DIS) tác động thế nào đến giá nhà (MEDV)?
- **H0:** Hệ số góc $\beta_1 = 0$ (Khoảng cách DIS không ảnh hưởng đến MEDV).
- **H1:** Hệ số góc $\beta_1 \neq 0$ (Khoảng cách DIS có ảnh hưởng đến MEDV).

In [ ]:
X = boston_df['DIS']
y = boston_df['MEDV']
X = sm.add_constant(X)

reg_model = sm.OLS(y, X).fit()
print(reg_model.summary())